In [1]:
import sqlite3
import os

# ===== 設定 =====
user_base = os.path.join(os.environ["USERPROFILE"], "myenv", "domz")
db_path = os.path.join(user_base, "db", "output.db")
columns_file = os.path.join(user_base, "db", "columns_with_type.txt")
table_name = "result_table"

conn = sqlite3.connect(db_path)
cur = conn.cursor()

# ✅ columns_with_type.txt 読み込み（順番通り）
with open(columns_file, encoding='utf-8') as f:
    cols_types = [line.strip().split() for line in f if line.strip()]
columns = [col for col, _ in cols_types]
types = [dtype for _, dtype in cols_types]

# ✅ 既存カラム確認
cur.execute(f"PRAGMA table_info({table_name});")
existing_cols = {row[1] for row in cur.fetchall()}

# ✅ 新テーブル作成SQL
col_defs = ", ".join(f"`{c}` {t}" for c, t in zip(columns, types))
create_sql = f"CREATE TABLE {table_name}_new ({col_defs});"
cur.execute(create_sql)
print(f"[INFO] ✅ 新テーブル {table_name}_new 作成")

# ✅ 共通カラムのみコピー
common_cols = [c for c in columns if c in existing_cols]
common_cols_str = ", ".join(f"`{c}`" for c in common_cols)

insert_sql = f'''
INSERT INTO {table_name}_new ({common_cols_str})
SELECT {common_cols_str} FROM {table_name};
'''
cur.execute(insert_sql)
print(f"[INFO] ✅ データ移行完了（{len(common_cols)}カラム）")

# ✅ 旧テーブル削除 & リネーム
cur.execute(f"DROP TABLE {table_name};")
cur.execute(f"ALTER TABLE {table_name}_new RENAME TO {table_name};")
print(f"[INFO] ✅ テーブル順番並び替え完了")

conn.commit()
conn.close()


[INFO] ✅ 新テーブル result_table_new 作成
[INFO] ✅ データ移行完了（439カラム）
[INFO] ✅ テーブル順番並び替え完了
